In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_glomerulus_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis"


os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_FOLDER,
    "gbm_glomerulus_stratification.csv"
)

OUTPUT_FIGURE = os.path.join(
    OUTPUT_FOLDER,
    "glomerulus_thickness_stratification.png"
)

if not os.path.exists(INPUT_CSV):

    raise FileNotFoundError(
        "\nInput file not found:\n"
        + INPUT_CSV
    )

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "patient_id",
    "glomerulus_id",
    "median_thickness_nm"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )

print()
print("GBM GLOMERULUS THICKNESS STRATIFICATION")
print("=" * 60)

print(
    f"Glomeruli analysed : {len(df)}"
)

print(
    f"Patients analysed  : "
    f"{df['patient_id'].nunique()}"
)

df = df.dropna(
    subset=["median_thickness_nm"]
).copy()

q33 = df["median_thickness_nm"].quantile(
    1 / 3
)

q67 = df["median_thickness_nm"].quantile(
    2 / 3
)

print()
print("THICKNESS STRATIFICATION THRESHOLDS")
print("-" * 60)

print(
    f"Lower threshold  : {q33:.2f} nm"
)

print(
    f"Upper threshold  : {q67:.2f} nm"
)

def classify_thickness(value):

    if value <= q33:

        return "Thin"

    elif value <= q67:

        return "Intermediate"

    else:

        return "Thick"


df["thickness_group"] = (
    df["median_thickness_nm"]
    .apply(classify_thickness)
)

group_order = [
    "Thin",
    "Intermediate",
    "Thick"
]

df["thickness_group"] = pd.Categorical(
    df["thickness_group"],
    categories=group_order,
    ordered=True
)

global_summary = (
    df.groupby(
        "thickness_group",
        observed=True
    )
    .agg(
        number_of_glomeruli=(
            "glomerulus_id",
            "count"
        ),

        mean_thickness_nm=(
            "median_thickness_nm",
            "mean"
        ),

        median_thickness_nm=(
            "median_thickness_nm",
            "median"
        ),

        std_thickness_nm=(
            "median_thickness_nm",
            "std"
        ),

        minimum_thickness_nm=(
            "median_thickness_nm",
            "min"
        ),

        maximum_thickness_nm=(
            "median_thickness_nm",
            "max"
        )
    )
    .reset_index()
)

global_summary[
    "std_thickness_nm"
] = global_summary[
    "std_thickness_nm"
].fillna(0)

print()
print("GLOBAL THICKNESS STRATIFICATION")
print("=" * 70)

print(
    global_summary.to_string(
        index=False
    )
)

patient_group_summary = (
    df.groupby(
        [
            "patient_id",
            "thickness_group"
        ],
        observed=True
    )
    .agg(
        number_of_glomeruli=(
            "glomerulus_id",
            "count"
        ),

        mean_thickness_nm=(
            "median_thickness_nm",
            "mean"
        ),

        median_thickness_nm=(
            "median_thickness_nm",
            "median"
        ),

        std_thickness_nm=(
            "median_thickness_nm",
            "std"
        )
    )
    .reset_index()
)

patient_group_summary[
    "std_thickness_nm"
] = patient_group_summary[
    "std_thickness_nm"
].fillna(0)


patient_totals = (
    df.groupby(
        "patient_id"
    )
    .size()
    .rename(
        "total_glomeruli"
    )
)

patient_group_summary = (
    patient_group_summary
    .merge(
        patient_totals,
        on="patient_id",
        how="left"
    )
)

patient_group_summary[
    "percentage_within_patient"
] = (
    patient_group_summary[
        "number_of_glomeruli"
    ]
    /
    patient_group_summary[
        "total_glomeruli"
    ]
    * 100
)


df.to_csv(
    OUTPUT_CSV,
    index=False
)

PATIENT_GROUP_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "gbm_patient_thickness_groups.csv"
)

patient_group_summary.to_csv(
    PATIENT_GROUP_OUTPUT,
    index=False
)

GLOBAL_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "gbm_global_thickness_groups.csv"
)

global_summary.to_csv(
    GLOBAL_OUTPUT,
    index=False
)

print()
print("Results saved:")
print(
    OUTPUT_CSV
)

print(
    PATIENT_GROUP_OUTPUT
)

print(
    GLOBAL_OUTPUT
)


fig, ax = plt.subplots(
    figsize=(10, 7)
)

data = []

for group in group_order:

    group_values = (
        df[
            df["thickness_group"] == group
        ]["median_thickness_nm"]
        .dropna()
        .values
    )

    data.append(group_values)

ax.boxplot(
    data,
    labels=group_order,
    showfliers=True
)

for i, values in enumerate(
    data,
    start=1
):

    if len(values) == 0:
        continue

    jitter = np.random.normal(
        i,
        0.04,
        size=len(values)
    )

    ax.scatter(
        jitter,
        values,
        alpha=0.65,
        s=28
    )

ax.set_xlabel(
    "GBM thickness group",
    fontsize=12
)

ax.set_ylabel(
    "Glomerular median GBM thickness (nm)",
    fontsize=12
)

ax.set_title(
    "Glomerulus Stratification by GBM Thickness",
    fontsize=14,
    fontweight="bold"
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

explanation = (
    "How to read this plot\n"
    "• Each point = one glomerulus\n"
    "• Box = middle 50% of glomerular values (IQR)\n"
    "• Centre line = median\n"
    "• Whiskers = 1.5 × IQR\n"
    "• Points beyond whiskers = potential outliers\n\n"
    "Groups are defined from the overall\n"
    "distribution of glomerular GBM thickness:\n"
    "• Thin = lowest third\n"
    "• Intermediate = middle third\n"
    "• Thick = highest third"
)

ax.text(
    1.03,
    0.98,
    explanation,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.9
    )
)

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print()
print("Figure saved:")
print(OUTPUT_FIGURE)

counts = (
    df["thickness_group"]
    .value_counts()
    .reindex(group_order)
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.bar(
    group_order,
    counts.values
)

ax.set_xlabel(
    "GBM thickness group",
    fontsize=12
)

ax.set_ylabel(
    "Number of glomeruli",
    fontsize=12
)

ax.set_title(
    "Number of Glomeruli in Each GBM Thickness Group",
    fontsize=14,
    fontweight="bold"
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

for i, value in enumerate(
    counts.values
):

    ax.text(
        i,
        value + max(counts.values) * 0.02,
        str(value),
        ha="center",
        fontsize=11
    )

COUNT_FIGURE = os.path.join(
    OUTPUT_FOLDER,
    "glomerulus_thickness_group_counts.png"
)

plt.tight_layout()

plt.savefig(
    COUNT_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print()
print("Group count figure saved:")
print(COUNT_FIGURE)


print()
print("FINAL STRATIFICATION SUMMARY")
print("=" * 60)

for group in group_order:

    n = (
        df[
            df["thickness_group"] == group
        ].shape[0]
    )

    percentage = (
        n / len(df) * 100
    )

    print(
        f"{group:<15} : "
        f"{n:>3} glomeruli "
        f"({percentage:.1f}%)"
    )

print()
print(
    f"Thin threshold         : "
    f"{q33:.2f} nm"
)

print(
    f"Intermediate threshold : "
    f"{q67:.2f} nm"
)

print()
print("GLOMERULUS STRATIFICATION COMPLETED")
print("=" * 60)


GBM GLOMERULUS THICKNESS STRATIFICATION
Glomeruli analysed : 257
Patients analysed  : 11

THICKNESS STRATIFICATION THRESHOLDS
------------------------------------------------------------
Lower threshold  : 153.26 nm
Upper threshold  : 228.60 nm

GLOBAL THICKNESS STRATIFICATION
thickness_group  number_of_glomeruli  mean_thickness_nm  median_thickness_nm  std_thickness_nm  minimum_thickness_nm  maximum_thickness_nm
           Thin                   86         126.964347           127.341313         17.150467             82.369098            153.254463
   Intermediate                   85         187.119070           190.684919         20.949186            153.271387            227.709030
          Thick                   86         374.638928           321.075121        166.393289            229.040135           1342.938095

Results saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_glomerulus_stratification.csv
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm

C:\Users\ishin\AppData\Local\Temp\ipykernel_7620\1333080871.py:304: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(



Figure saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\glomerulus_thickness_stratification.png

Group count figure saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\glomerulus_thickness_group_counts.png

FINAL STRATIFICATION SUMMARY
Thin            :  86 glomeruli (33.5%)
Intermediate    :  85 glomeruli (33.1%)
Thick           :  86 glomeruli (33.5%)

Thin threshold         : 153.26 nm
Intermediate threshold : 228.60 nm

GLOMERULUS STRATIFICATION COMPLETED
